# Task 1.2b: Data Cleaning & Standardization

## Overview
This notebook handles the data cleaning and standardization phase of the Student-Place Preference Prediction System. The main objectives are:

1. **Handle Missing Data**: Identify and handle missing values in the dataset
2. **Standardize Text**: Normalize text fields (lowercase, trim spaces)
3. **Map Categories**: Standardize categorical values:
   - **Time of Day**: Map to 4 classes (Morning, Afternoon, Evening, Night)
   - **Country**: Standardize country names (handle cities, regions, typos)
4. **Add Preference Labels**: Create binary preference labels (1 for student-selected places, 0 for others)

## Expected Deliverables
- Cleaned dataset: `data/processed/cleaned_dataset.csv`
- Statistics on data quality and category distributions
- Documentation of mapping decisions

## 1. Import Libraries

In [11]:
import pandas as pd 
import numpy as np
import os
from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load Dataset

In [12]:
# Load the dataset
df = pd.read_csv('../data/processed/master_dataset.csv')

print(f"Dataset loaded: {len(df)} records")
print(f"\nColumns: {list(df.columns)}")
print(f"\nDataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

Dataset loaded: 1076 records

Columns: ['Image URL', 'Description', 'Country', 'Weather', 'Time of Day', 'Season', 'Activity', 'Mood/Emotion', 'source_file']

Dataset shape: (1076, 9)

First few rows:


,Image URL,Description,Country,Weather,Time of Day,Season,Activity,Mood/Emotion,source_file
0,https://commons.wikimedia.org/wiki/File:Dom_of...,A clear image of the Dome of the Rock in Jerus...,Palestine,Sunny,Afternoon,Summer,Sightseeing,Nostalgia,2936035-1161937.csv
1,https://upload.wikimedia.org/wikipedia/commons...,a clear image of the Ibrahimi Mosque (Cave of ...,Palestine,Sunny,Morning,Spring,Sightseeing,Curiosity,2936035-1161937.csv
2,https://upload.wikimedia.org/wikipedia/commons...,A clear image of the ancient ruins in Sebastia...,Palestine,Sunny,Afternoon,Summer,Exploring,Adventure,2936035-1161937.csv
3,https://upload.wikimedia.org/wikipedia/commons...,A clear image of Mar Saba Monastery in Bethleh...,Palestine,Sunny,Afternoon,Summer,Sightseeing,Awe,2936035-1161937.csv
4,https://upload.wikimedia.org/wikipedia/commons...,A clear aerial view of Tell es-Sultan in Jeric...,Palestine,Sunny,Morning,Spring,Exploring,Curiosity,2936035-1161937.csv


## 3. Check for Missing Values

In [13]:
# Check for missing values
print("Missing values in each column:")
print(df.isnull().sum())
print("\nData types of each column:")
print(df.dtypes)
print("\nBasic statistics:")
print(df.describe(include='all'))

Missing values in each column:
Image URL        0
Description      0
Country          1
Weather          0
Time of Day      0
Season           0
Activity        22
Mood/Emotion     1
source_file      0
dtype: int64

Data types of each column:
Image URL       object
Description     object
Country         object
Weather         object
Time of Day     object
Season          object
Activity        object
Mood/Emotion    object
source_file     object
dtype: object

Basic statistics:
                                                Image URL  \
count                                                1076   
unique                                                955   
top     https://www.sunsiyam.com/media/qnfnzgmq/ssiv_g...   
freq                                                    6   

                                              Description Country Weather  \
count                                                1076    1075    1076   
unique                                                959

## 4. Text Standardization
Convert all text columns to lowercase and remove extra whitespace.

In [14]:
# Convert text columns to lowercase and remove extra whitespace
text_columns = ['Country', 'Time of Day', 'Description']

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().str.strip()

print("Text standardization complete.")
print(f"\nUnique countries before mapping: {df['Country'].nunique()}")
print(f"Unique time of day values before mapping: {df['Time of Day'].nunique()}")

Text standardization complete.

Unique countries before mapping: 132
Unique time of day values before mapping: 13


## 5. Time of Day Mapping

Map all time values to **4 main categories**:

1. **Morning** — early morning, dawn, sunrise, morning
2. **Afternoon** — midday, noon, afternoon
3. **Evening** — sunset, dusk, evening, twilight
4. **Night** — night, nighttime, late night

In [15]:
def map_time_of_day(df, time_column='Time of Day'):
    """
    Maps time of day values to 4 standardized categories.
    
    Parameters:
    df (pd.DataFrame): DataFrame containing a time of day column
    time_column (str): Name of the column containing time data
    
    Returns:
    pd.DataFrame: DataFrame with a new 'Time_of_Day_Standardized' column
    """
    
    # Define the mapping dictionary
    time_mapping = {
        'Morning': [
            'morning', 'early morning', 'dawn', 'sunrise', 'early',
            'morning.', 'morn', 'am', 'breakfast time'
        ],
        
        'Afternoon': [
            'afternoon', 'midday', 'noon', 'mid-day', 'lunch time',
            'afternoon.', 'pm', 'daytime', 'day time'
        ],
        
        'Evening': [
            'evening', 'sunset', 'dusk', 'twilight', 'evening.',
            'late afternoon', 'golden hour', 'sunsetting'
        ],
        
        'Night': [
            'night', 'nighttime', 'late night', 'midnight', 'night.',
            'night time', 'late', 'after dark', 'dark'
        ]
    }
    
    # Create reverse mapping (time variant -> standard category)
    reverse_mapping = {}
    for category, variants in time_mapping.items():
        for variant in variants:
            reverse_mapping[variant.lower()] = category
    
    # Function to map individual time value
    def map_single_time(time_value):
        if pd.isna(time_value):
            return 'Unknown'
        
        time_lower = str(time_value).lower().strip()
        
        # Handle unclear/invalid values
        if time_lower in ['not clear', 'nan', '', 'none']:
            return 'Invalid'
        
        # Direct match
        if time_lower in reverse_mapping:
            return reverse_mapping[time_lower]
        
        # Partial match for compound entries
        for key, category in reverse_mapping.items():
            if key in time_lower or time_lower in key:
                return category
        
        return 'Other'
    
    # Apply mapping
    df['Time_of_Day_Standardized'] = df[time_column].apply(map_single_time)
    
    return df

# Apply time of day mapping
df = map_time_of_day(df)

# Display results
print("Sample Time of Day Mappings:")
print(df[['Time of Day', 'Time_of_Day_Standardized']].head(20))

print("\n=== Time of Day Distribution ===" )
print(df['Time_of_Day_Standardized'].value_counts())

print("\n=== Time of Day Percentage ====")
print(df['Time_of_Day_Standardized'].value_counts(normalize=True) * 100)

Sample Time of Day Mappings:
   Time of Day Time_of_Day_Standardized
0    afternoon                Afternoon
1      morning                  Morning
2    afternoon                Afternoon
3    afternoon                Afternoon
4      morning                  Morning
5    afternoon                Afternoon
6      morning                  Morning
7      evening                  Evening
8      evening                  Evening
9    afternoon                Afternoon
10     morning                  Morning
11   afternoon                Afternoon
12     morning                  Morning
13   afternoon                Afternoon
14   afternoon                Afternoon
15     evening                  Evening
16     morning                  Morning
17     evening                  Evening
18   afternoon                Afternoon
19   afternoon                Afternoon

=== Time of Day Distribution ===
Time_of_Day_Standardized
Afternoon    499
Evening      295
Morning      252
Night         20
Inva

## 6. Country Standardization

Standardize country names to handle:
- Cities mapped to their countries (e.g., Paris → France)
- Alternate spellings and typos
- Regional variations

In [16]:
def map_countries(df, country_column='Country'):
    """
    Maps country values to standardized country names.
    Handles cities, regions, alternate names, and typos.
    
    Parameters:
    df (pd.DataFrame): DataFrame containing a country column
    country_column (str): Name of the column containing country data
    
    Returns:
    pd.DataFrame: DataFrame with a new 'Country_Standardized' column
    """
    
    # Define the mapping dictionary
    country_mapping = {
        'Palestine': ['palestine'],
        'France': ['france', 'paris'],
        'China': ['china', 'china and mongolia'],
        'Greece': ['greece', 'santorini'],
        'USA': ['usa', 'united states', 'united states of america', 'miami', 
                'new york', 'california', 'california, usa', 'north pole, alaska'],
        'Maldives': ['maldives', 'maldive'],
        'Japan': ['japan', 'tokyo/japan'],
        'Peru': ['peru', 'preu'],
        'Canada': ['canada'],
        'Tanzania': ['tanzania'],
        'Australia': ['australia'],
        'Iceland': ['iceland'],
        'Jordan': ['jordan'],
        'Italy': ['italy', 'italy (lake como)'],
        'Austria': ['austria', 'vienna'],
        'Brazil': ['brazil'],
        'Vietnam': ['vietnam'],
        'Portugal': ['portugal'],
        'Switzerland': ['switzerland', 'interlaken'],
        'Turkey': ['turkey', 'türkiye', 'turkiye'],
        'Nepal': ['nepal'],
        'Egypt': ['egypt', 'eygypt'],
        'Indonesia': ['indonesia', 'bali'],
        'South Africa': ['south africa'],
        'Spain': ['spain'],
        'Greenland': ['greenland'],
        'Tunisia': ['tunisia'],
        'Syria': ['syria'],
        'Morocco': ['morocco', 'morroco'],
        'UK': ['uk', 'united kingdom', 'london', 'england', 'scotland', 
               'wales', 'united kingdom (scotland)', 'holland'],
        'Germany': ['germany'],
        'UAE': ['uae', 'united arab emirates'],
        'Netherlands': ['netherlands', 'the netherlands'],
        'Czech Republic': ['czech republic', 'czechia', 'prague'],
        'Slovenia': ['slovenia'],
        'Chile': ['chile', 'easter island', 'rapa nui'],
        'Saudi Arabia': ['saudi arabia', 'mecca', 'suadi arabia'],
        'India': ['india'],
        'South Korea': ['south korea', 'korea'],
        'Thailand': ['thailand'],
        'Cambodia': ['cambodia'],
        'Antarctica': ['antarctica', 'antarctica (continent, not a country)'],
        'Mexico': ['mexico'],
        'Monaco': ['monaco'],
        'Denmark': ['denmark'],
        'Qatar': ['qatar'],
        'Finland': ['finland'],
        'Myanmar': ['myanmar'],
        'New Zealand': ['new zealand'],
        'Hungary': ['hungary'],
        'Bhutan': ['bhutan'],
        'Poland': ['poland'],
        'Andorra': ['andorra'],
        'Cyprus': ['cyprus'],
        'Kenya': ['kenya'],
        'Georgia': ['georgia'],
        'Malta': ['malta'],
        'Lebanon': ['lebanon'],
        'Hong Kong': ['hong kong'],
        'Russia': ['russia'],
        'Yemen': ['yemen'],
        'Zimbabwe': ['zimbabwe', 'zimbabwe/zambia'],
        'Singapore': ['singapore'],
        'Algeria': ['algeria'],
        'Zambia': ['zambia'],
        'Oman': ['oman'],
        'Uzbekistan': ['uzbekistan'],
        'Seychelles': ['seychelles'],
        'Sweden': ['sweden'],
        'Bosnia and Herzegovina': ['bosnia and herzegovina'],
        'Kazakhstan': ['kazakhstan'],
        'Ireland': ['ireland'],
        'Albania': ['albania'],
        'Malaysia': ['malaysia'],
        'Pakistan': ['pakistan'],
        'Croatia': ['croatia'],
        'Bahamas': ['bahamas'],
        'Argentina': ['argentina'],
        'Afghanistan': ['afghanistan'],
        'Philippines': ['philippines'],
        'Bolivia': ['bolivia'],
        'Ukraine': ['ukraine'],
        'Serbia': ['serbia'],
        'Colombia': ['colombia'],
        'Taiwan': ['taiwan'],
        'Norway': ['norway'],
        'French Polynesia': ['french polynesia'],
        'Arctic': ['arctic circle']
    }
    
    # Create reverse mapping (variant -> standard name)
    reverse_mapping = {}
    for standard_name, variants in country_mapping.items():
        for variant in variants:
            reverse_mapping[variant.lower()] = standard_name
    
    # Function to map individual country
    def map_single_country(country):
        if pd.isna(country):
            return 'Unknown'
        
        country_lower = str(country).lower().strip()
        
        # Handle invalid/technical values
        if country_lower in ['c_auto', 'w_3840', 'c_fill', 'nan']:
            return 'Invalid'
        
        # Direct match
        if country_lower in reverse_mapping:
            return reverse_mapping[country_lower]
        
        # Partial match for compound entries
        for key, standard in reverse_mapping.items():
            if key in country_lower:
                return standard
        
        return 'Other'
    
    # Apply mapping
    df['Country_Standardized'] = df[country_column].apply(map_single_country)
    
    return df

# Apply country mapping
df = map_countries(df, country_column='Country')

# Display sample mappings
print("Sample Country Mappings:")
print(df[['Country', 'Country_Standardized']].head(20))

print("\n=== Country Mapping Statistics ===")
print(f"Total records: {len(df)}")
print(f"Unique original countries: {df['Country'].nunique()}")
print(f"Unique standardized countries: {df['Country_Standardized'].nunique()}")
print(f"Unknown count: {(df['Country_Standardized'] == 'Unknown').sum()}")
print(f"Invalid count: {(df['Country_Standardized'] == 'Invalid').sum()}")
print(f"Other count: {(df['Country_Standardized'] == 'Other').sum()}")

# Show distribution
print("\n=== Top 20 Countries ===")
print(df['Country_Standardized'].value_counts().head(20))

Sample Country Mappings:
      Country Country_Standardized
0   palestine            Palestine
1   palestine            Palestine
2   palestine            Palestine
3   palestine            Palestine
4   palestine            Palestine
5      france               France
6       china                China
7      greece               Greece
8         usa                  USA
9    maldives             Maldives
10      japan                Japan
11     greece               Greece
12       peru                 Peru
13     canada               Canada
14   tanzania             Tanzania
15        usa                  USA
16  australia            Australia
17    iceland              Iceland
18     jordan               Jordan
19      italy                Italy

=== Country Mapping Statistics ===
Total records: 1076
Unique original countries: 132
Unique standardized countries: 89
Unknown count: 0
Invalid count: 4
Other count: 0

=== Top 20 Countries ===
Country_Standardized
Italy           71
Japa

## 7. Add Preference Labels

Since each student has their selected places in the dataset, we'll mark all current records as preference=1 (student selected).
Later, in Task 1.4 (training pairs), we'll create negative samples from other students' places.

In [17]:
# Add preference column - all current records are student-selected (positive examples)
df['preference'] = 1

print(f"Added preference column: {df['preference'].value_counts().to_dict()}")
print(f"\nAll {len(df)} records are labeled as preferred (1).")
print("\nNegative samples will be created in Task 1.4 during training pair generation.")

Added preference column: {1: 1076}

All 1076 records are labeled as preferred (1).

Negative samples will be created in Task 1.4 during training pair generation.


## 8. Clean Up and Handle Missing/Invalid Data

Remove records with invalid or missing critical information.

In [18]:
# Before cleanup
initial_count = len(df)
print(f"Initial record count: {initial_count}")

# Check for records with Invalid/Unknown values
print("\n=== Records with Invalid/Unknown Values ===")
print(f"Country Unknown/Invalid: {((df['Country_Standardized'] == 'Unknown') | (df['Country_Standardized'] == 'Invalid')).sum()}")
print(f"Time of Day Unknown/Invalid: {((df['Time_of_Day_Standardized'] == 'Unknown') | (df['Time_of_Day_Standardized'] == 'Invalid')).sum()}")

# Option 1: Keep all records (handle Unknown/Invalid in modeling)
# Option 2: Remove records with Unknown/Invalid critical fields

# For now, let's keep all records and report on data quality
# We can handle Unknown/Invalid during model training

# Remove duplicates based on Image URL (if any)
df_cleaned = df.drop_duplicates(subset=['Image URL'], keep='first')
duplicates_removed = initial_count - len(df_cleaned)

print(f"\nDuplicates removed: {duplicates_removed}")
print(f"Final record count: {len(df_cleaned)}")

df = df_cleaned

Initial record count: 1076

=== Records with Invalid/Unknown Values ===
Country Unknown/Invalid: 4
Time of Day Unknown/Invalid: 7

Duplicates removed: 121
Final record count: 955


## 9. Final Dataset Statistics

In [19]:
print("=" * 60)
print("FINAL DATASET STATISTICS")
print("=" * 60)

print(f"\nTotal Records: {len(df)}")
print(f"Total Students: {df['source_file'].nunique()}")

print("\n--- Time of Day Distribution ---")
time_dist = df['Time_of_Day_Standardized'].value_counts()
time_pct = df['Time_of_Day_Standardized'].value_counts(normalize=True) * 100
for category in time_dist.index:
    print(f"{category}: {time_dist[category]} ({time_pct[category]:.2f}%)")

print("\n--- Country Distribution (Top 15) ---")
country_dist = df['Country_Standardized'].value_counts().head(15)
country_pct = df['Country_Standardized'].value_counts(normalize=True).head(15) * 100
for country in country_dist.index:
    print(f"{country}: {country_dist[country]} ({country_pct[country]:.2f}%)")

print("\n--- Data Quality Metrics ---")
print(f"Missing Image URLs: {df['Image URL'].isnull().sum()}")
print(f"Missing Descriptions: {df['Description'].isnull().sum()}")
print(f"Unknown Countries: {(df['Country_Standardized'] == 'Unknown').sum()}")
print(f"Invalid Countries: {(df['Country_Standardized'] == 'Invalid').sum()}")
print(f"Unknown Time of Day: {(df['Time_of_Day_Standardized'] == 'Unknown').sum()}")
print(f"Invalid Time of Day: {(df['Time_of_Day_Standardized'] == 'Invalid').sum()}")

print("\n--- Preference Distribution ---")
print(df['preference'].value_counts())

FINAL DATASET STATISTICS

Total Records: 955
Total Students: 100

--- Time of Day Distribution ---
Afternoon: 442 (46.28%)
Evening: 260 (27.23%)
Morning: 225 (23.56%)
Night: 18 (1.88%)
Invalid: 7 (0.73%)
Other: 3 (0.31%)

--- Country Distribution (Top 15) ---
Italy: 65 (6.81%)
Japan: 62 (6.49%)
USA: 57 (5.97%)
France: 50 (5.24%)
Switzerland: 49 (5.13%)
Spain: 49 (5.13%)
Palestine: 44 (4.61%)
Maldives: 35 (3.66%)
Greece: 35 (3.66%)
Turkey: 34 (3.56%)
UK: 32 (3.35%)
Egypt: 29 (3.04%)
Saudi Arabia: 26 (2.72%)
UAE: 23 (2.41%)
China: 23 (2.41%)

--- Data Quality Metrics ---
Missing Image URLs: 0
Missing Descriptions: 0
Unknown Countries: 0
Invalid Countries: 4
Unknown Time of Day: 0
Invalid Time of Day: 7

--- Preference Distribution ---
preference
1    955
Name: count, dtype: int64


## 10. Save Cleaned Dataset

In [20]:
# Select relevant columns for the cleaned dataset
cleaned_columns = [
    'source_file',  # Student identifier
    'Image URL',
    'Description',
    'Country',
    'Country_Standardized',
    'Time of Day',
    'Time_of_Day_Standardized',
    'preference'
]

# Keep only columns that exist in the dataframe
existing_columns = [col for col in cleaned_columns if col in df.columns]
df_final = df[existing_columns]

# Save to CSV
output_path = '../data/processed/cleaned_dataset.csv'
df_final.to_csv(output_path, index=False)

print(f"✓ Cleaned dataset saved to: {output_path}")
print(f"✓ Total records: {len(df_final)}")
print(f"✓ Total columns: {len(df_final.columns)}")
print(f"\nColumns in cleaned dataset:")
for col in df_final.columns:
    print(f"  - {col}")

✓ Cleaned dataset saved to: ../data/processed/cleaned_dataset.csv
✓ Total records: 955
✓ Total columns: 8

Columns in cleaned dataset:
  - source_file
  - Image URL
  - Description
  - Country
  - Country_Standardized
  - Time of Day
  - Time_of_Day_Standardized
  - preference


## 11. Summary and Next Steps

### Completed:
✅ Loaded master dataset  
✅ Standardized Time of Day into 4 categories (Morning, Afternoon, Evening, Night)  
✅ Standardized Country names (handling cities, typos, variations)  
✅ Added preference labels (all current records = 1)  
✅ Cleaned and removed duplicates  
✅ Saved cleaned dataset  

### Next Steps:
1. **Task 1.3**: Create train/val/test splits by students
2. **Task 1.4**: Generate training pairs with negative samples
3. **Task 1.5**: Perform EDA on the cleaned dataset

### Key Insights for Report:
- Dataset contains student preferences across multiple countries and time periods
- Time of day and country are well-distributed auxiliary features
- Clean, standardized data ready for modeling